# SERS-D2DNet graph and supervised targets

## Goal
Inspect the supplied graph interface using a narrow synthetic verification profile.

## Setup
Install the package in this kernel's environment. All default data are synthetic; research training is disabled.

In [ ]:
from pathlib import Path
import os, sys, uuid
import numpy as np
PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()), Path.cwd())
WORK = Path(os.environ.get("SERS_NOTEBOOK_OUTPUT", PROJECT / "output/notebooks")) / uuid.uuid4().hex[:8]
WORK.mkdir(parents=True)
from sersd2d_superraman.data import SpectrumBatch, load_batch, save_batch
from sersd2d_superraman.cli import synthetic, prepare

### 1. Build a bounded graph

In [ ]:
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
from sersd2d_superraman.translation.model import build_model, TranslationConfig, primary_output
from sersd2d_superraman.translation.training import supervision_targets
config = TranslationConfig(width=2)
model = build_model(config)
synthetic(WORK / "synthetic", per_class=5)
batch = load_batch(WORK / "synthetic/Enwave.npz")
{ "architecture": config.architecture_id, "parameters": model.count_params(), "output_names": model.output_names }

### 2. Check the output and supervision

In [ ]:
translated = primary_output(model, batch.spectra[:2], batch_size=1)
targets = supervision_targets(batch.spectra[:2])
assert translated.shape == (2,1400) and np.isfinite(translated).all()
{k: value.shape for k, value in targets.items()}

### 3. Research training command

In [ ]:
RUN_RESEARCH_TRAINING = False
command = [sys.executable, "-m", "sersd2d_superraman", "train-translation", "--data", "data/prepared", "--source", "Enwave", "--config", "configs/maintained/translation.json", "--output", "output/enwave"]
if RUN_RESEARCH_TRAINING:
    import subprocess
    subprocess.run(command, cwd=PROJECT, check=True)
else:
    print("Prepared command:", " ".join(command))

## Checks and next steps
The research default is depth 3/width 128 and 61,511,562 parameters. The narrow graph above is a software check. Recover paper weights and resolve the reported 6.15 M discrepancy before certifying paper reproduction.